In [13]:
import numpy as np
import pandas as pd

In [14]:
df= pd.read_csv('rsi_monthly_raw.csv')

In [15]:
print(df.shape)
print(df.dtypes)
df.head

(10912, 5)
Statistic Label     object
Month               object
NACE Group          object
UNIT                object
VALUE              float64
dtype: object


<bound method NDFrame.head of                            Statistic Label          Month  \
0       Retail Sales Index Volume Adjusted   2021 January   
1       Retail Sales Index Volume Adjusted   2021 January   
2       Retail Sales Index Volume Adjusted   2021 January   
3       Retail Sales Index Volume Adjusted   2021 January   
4       Retail Sales Index Volume Adjusted   2021 January   
...                                    ...            ...   
10907  Retail Sales Index Value Unadjusted  2026 February   
10908  Retail Sales Index Value Unadjusted  2026 February   
10909  Retail Sales Index Value Unadjusted  2026 February   
10910  Retail Sales Index Value Unadjusted  2026 February   
10911  Retail Sales Index Value Unadjusted  2026 February   

                                              NACE Group                UNIT  \
0                                      Motor trades (45)  Base Year 2021=100   
1      Retail sale in non-specialised stores with foo...  Base Year 2021=100 

In [16]:
# Rename to clean, consistent snake_case names 
df.columns = ['statistic', 'month_raw', 'nace_group', 'unit', 'value']

# Confirm the rename worked
print(df.columns.tolist())

['statistic', 'month_raw', 'nace_group', 'unit', 'value']


In [17]:
# UNIT column has no useful information - drop it
df = df.drop(columns=['unit'])

# confirm it's gone
print(df.columns.tolist())
print(df.shape)

['statistic', 'month_raw', 'nace_group', 'value']
(10912, 4)


In [18]:
# Keep only the two adjusted index series
keep_statistics = [
    'Retail Sales Index Value Adjusted',
    'Retail Sales Index Volume Adjusted'
]

df = df[df['statistic'].isin(keep_statistics)]

# Check how many rows remain
print(df.shape)
print(df['statistic'].value_counts())

(2728, 4)
statistic
Retail Sales Index Volume Adjusted    1364
Retail Sales Index Value Adjusted     1364
Name: count, dtype: int64


In [19]:
# The 13 individual sectors plus the all-retail aggregate
keep_sectors = [
    'All retail businesses',
    'Motor trades (45)',
    'Retail sale of automotive fuel (4730)',
    'Retail sale in non-specialised stores with food, beverages or tobacco predominating (4711)',
    'Retail sale of food, beverages and tobacco in specialised stores (4721 to 4729)',
    'Retail sale of textiles, clothing and footwear (4751,4771,4772)',
    'Retail sale of hardware, paints and glass (4752)',
    'Retail sale of electrical goods (4741 to 4743,4754)',
    'Retail sale of furniture and lighting (4759)',
    'Retail sale of pharmaceutical, medical and cosmetic articles (4773 to 4775)',
    'Department stores (4719)',
    'Retail sale of books, newspapers and stationery (4761,4762)',
    'Other retail sales (4753,4763 to 4765,4776 to 4778)',
    'Bars (5630)'
]

df = df[df['nace_group'].isin(keep_sectors)]

# Check
print(df.shape)
print(df['nace_group'].nunique(), 'sectors remaining')

(1736, 4)
14 sectors remaining


In [20]:
# Map long NACE names to short readable labels
sector_names = {
    'All retail businesses': 'All Retail',
    'Motor trades (45)': 'Motor Trades',
    'Retail sale of automotive fuel (4730)': 'Automotive Fuel',
    'Retail sale in non-specialised stores with food, beverages or tobacco predominating (4711)': 'Food Non-Specialised',
    'Retail sale of food, beverages and tobacco in specialised stores (4721 to 4729)': 'Food Specialised',
    'Retail sale of textiles, clothing and footwear (4751,4771,4772)': 'Clothing & Footwear',
    'Retail sale of hardware, paints and glass (4752)': 'Hardware & Glass',
    'Retail sale of electrical goods (4741 to 4743,4754)': 'Electrical Goods',
    'Retail sale of furniture and lighting (4759)': 'Furniture & Lighting',
    'Retail sale of pharmaceutical, medical and cosmetic articles (4773 to 4775)': 'Pharmaceuticals',
    'Department stores (4719)': 'Department Stores',
    'Retail sale of books, newspapers and stationery (4761,4762)': 'Books & Stationery',
    'Other retail sales (4753,4763 to 4765,4776 to 4778)': 'Other Retail',
    'Bars (5630)': 'Bars'
}

df['sector'] = df['nace_group'].map(sector_names)

# Confirm no nulls in the new column - should print 0
print('Unmapped sectors:', df['sector'].isna().sum())
df[['nace_group', 'sector']].drop_duplicates()

Unmapped sectors: 0


,nace_group,sector
0,Motor trades (45),Motor Trades
1,Retail sale in non-specialised stores with foo...,Food Non-Specialised
2,Department stores (4719),Department Stores
3,Retail sale of automotive fuel (4730),Automotive Fuel
4,"Retail sale of hardware, paints and glass (4752)",Hardware & Glass
5,Retail sale of furniture and lighting (4759),Furniture & Lighting
6,Bars (5630),Bars
7,All retail businesses,All Retail
14,"Retail sale of food, beverages and tobacco in ...",Food Specialised
16,"Retail sale of electrical goods (4741 to 4743,...",Electrical Goods


In [22]:
# Convert "2021 January" format to a proper datetime
df['month'] = pd.to_datetime(df['month_raw'], format='%Y %B')

# Confirm the conversion
print(df['month'].dtype)
print(df['month'].min(), 'to', df['month'].max())
df[['month_raw', 'month']].drop_duplicates().head(12)

datetime64[ns]
2021-01-01 00:00:00 to 2026-02-01 00:00:00


,month_raw,month
0,2021 January,2021-01-01
22,2021 February,2021-02-01
44,2021 March,2021-03-01
66,2021 April,2021-04-01
88,2021 May,2021-05-01
110,2021 June,2021-06-01
132,2021 July,2021-07-01
154,2021 August,2021-08-01
176,2021 September,2021-09-01
198,2021 October,2021-10-01


In [23]:
# Separate the two series
value_df = df[df['statistic'] == 'Retail Sales Index Value Adjusted'][['month', 'sector', 'value']].copy()
volume_df = df[df['statistic'] == 'Retail Sales Index Volume Adjusted'][['month', 'sector', 'value']].copy()

# Rename value column before merging
value_df = value_df.rename(columns={'value': 'value_index'})
volume_df = volume_df.rename(columns={'value': 'volume_index'})

# Merge on month and sector
df_clean = pd.merge(value_df, volume_df, on = ['month', 'sector'], how='inner')

# Check the result 
print(df_clean.shape)
print(df_clean.dtypes)
df_clean.head()

(868, 4)
month           datetime64[ns]
sector                  object
value_index            float64
volume_index           float64
dtype: object


,month,sector,value_index,volume_index
0,2021-01-01,Motor Trades,72.8,83.5
1,2021-01-01,Food Non-Specialised,102.8,102.3
2,2021-01-01,Department Stores,68.0,65.1
3,2021-01-01,Automotive Fuel,69.9,77.7
4,2021-01-01,Hardware & Glass,86.3,89.8


In [26]:
# Check for any nulls in the clean dataset
print(df_clean.isnull().sum())

# Check ebery sectore has the same number of months
print(df_clean.groupby('sector')['month'].count().sort_values())

month           0
sector          0
value_index     0
volume_index    0
dtype: int64
sector
All Retail              62
Automotive Fuel         62
Bars                    62
Books & Stationery      62
Clothing & Footwear     62
Department Stores       62
Electrical Goods        62
Food Non-Specialised    62
Food Specialised        62
Furniture & Lighting    62
Hardware & Glass        62
Motor Trades            62
Other Retail            62
Pharmaceuticals         62
Name: month, dtype: int64


In [27]:
# Export to CSV for use in power BI
df_clean.to_csv('rsi_clean.csv', index=False)

print('Exported successfully')
print(f'Final shape: {df_clean.shape}')
print(f'Sectors: {df_clean["sector"].nunique()}')
print(f'Date range: {df_clean["month"].min().strftime("%B %Y")} to {df_clean["month"].max().strftime("%B %Y")}')

Exported successfully
Final shape: (868, 4)
Sectors: 14
Date range: January 2021 to February 2026
